# **Assignment: Fine-Tuning BERT for POS Tagging & Chunking**

In [ ]:
!pip install transformers[torch] datasets evaluate seqeval

IMPORTS

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np

## LOAD DATASET

In [ ]:
raw_datasets = load_dataset("lhoestq/conll2003")
raw_datasets

## LABEL CODE

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("lhoestq/conll2003")

# define label list (official CoNLL-2003 labels)
label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

label_list

## TOKENIZER CODE

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

## ALIGNMENT CODE

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

## APPLY TOKENIZATION

In [ ]:
tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True
)

## MODEL CODE

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

## TRAINING CODE

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

training_args = TrainingArguments(
    "ner-distilbert-assignment",
    eval_strategy="epoch",    # Modern strategy name
    save_strategy="epoch",
    learning_rate=2e-5,       # Mandatory Learning Rate
    per_device_train_batch_size=16,
    num_train_epochs=3,       # Mandatory Epochs
    weight_decay=0.01,
    load_best_model_at_end=True
)

data_collator = DataCollatorForTokenClassification(tokenizer)

## METRICS CODE

In [ ]:
import evaluate
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

## TRAIN MODEL

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(1500)),
    eval_dataset=tokenized_datasets["validation"].select(range(300)),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

## INFERENCE CODE

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer)

sentence = "Ganesh Mundal at Nashik completing B-Tech in computer science"
result = ner_pipeline(sentence)

# Clean output
for entity in result:
    print(f"Word: {entity['word']}, Label: {entity['entity']}, Score: {round(entity['score'], 2)}")

### Inference Observation

Due to WordPiece tokenization, some words are split into subwords (e.g., "Ganesh" → "gan", "##esh").  
To improve readability, aggregation strategy is used to merge subwords into complete entities.

## Comparison: POS Tagging vs Chunking

| Feature | POS Tagging | Chunking |
|--------|------------|----------|
| Level | Word-level | Phrase-level |
| Output | Grammatical tags | Entity groups |
| Difficulty | Easy | Medium |
| Example | Noun, Verb | Noun Phrase |

Chunking provides better semantic understanding compared to POS tagging.

## Challenges Faced

- Handling subword tokenization
- Aligning labels with tokens
- Managing -100 ignored labels

## Observations

- Transformer models perform well for sequence labeling
- Chunking gives more meaningful context than POS tagging